**Wind Power Model Example**

This script walks you through running a simple wind power model to recreate the sensitivity analysis workflow used in the manuscript 'Uncertainty quantification and attribution for resilient infrastructure systems' which has been submitted to the Climate Resilience and Sustainability journal by Salwey et al. 


Import python libraries 

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import scipy.stats as st
import xarray as xr

Import SAFEtoolbox functions for global sensitivity analysis 

In [ ]:
from safepython import PAWN
from safepython.sampling import AAT_sampling
from safepython.model_execution import model_execution
from safepython.util import aggregate_boot

Import script containing functions to run simple wind power model

In [ ]:
import simple_wind_power

Define the number of input parameters (M) in the sensitivity analysis and define their sampling ranges and distributions. 

In this example we have 6 input variables: Wind Height, Tiestep, Alpha, Power Curve, Hub-Height and Simulation Period. 

In [ ]:
# Define labels for the input (X) variables 
X_Labels = [ 'Wind \nHeight', 'Timestep', 'Alpha','Power \nCurve', 'Hub-\nheight', 'Simulation \nPeriod', 'dummy']

# Define the number of variables (M)
M = 6 

# Define the sampling bounds for each parameter:
xmin = [0, 0, 0.05, 0, 75, 1960]
xmax = [1, 3, 0.23, 8, 140, 2020]


# Define the parameter distributions:

# In this example all of the input variables have discrete distributions except for the Alpha and Hub-height variables which have continuous uniform distributions.

distr_fun = [st.randint] * M # discrete uniform distribution
distr_fun[-4] = st.uniform # uniform distribution
distr_fun[-2] = st.uniform

# The shape parameters of the discrete uniform distribution are the lower limit
# and the upper limit+1:
distr_par = [np.nan] * M
for i in range(M):
    distr_par[i] =[xmin[i], xmax[i] + 1] 
# The shape parameters of the uniform distribution are the lower limit and the
# difference between lower and upper limits:
distr_par[-4] = [xmin[-4], xmax[-4] - xmin[-4]]
distr_par[-2] = [xmin[-2], xmax[-2] - xmin[-2]]


Sample the input variables within the chosen bounds. 

In [ ]:
# Define the sampling strategy and number of samples

samp_strat = 'lhs' # Latin Hypercube
N = 2000  #  Number of samples

# Define the input dataset
X = AAT_sampling(samp_strat, M, distr_fun, distr_par, N)

We now need to define the function used to run the wind power model and create the output. 

In [ ]:
fun_run_model = simple_wind_power.wind_power_model_run

# First read in the power curves which are used to run the model 
power_curves = pd.read_csv('UK_offshore_power_curves.csv', index_col=0)


Y = model_execution(fun_run_model, X, power_curves)

Since some of our input variables have no physical meaning, we convert them to continuous reference values. E.g. the discrete power curves are represented by their power at a given point and the simulation period is represented by the average wind speed in the chosen year. 

In [ ]:
longitude = 1.91
latitude = 54.77

# Store mean wind speed for each year (computed once per year)

year_wind_speed = {}
quantile_wind_speed = {}

# Iterate over unique years in X[:,5] 
for year in np.unique(X[:, 5]):
    ds = xr.open_dataset(
        f'data/ERA5_EU_1hr_uv100m_{int(year)}.nc',
        combine='by_coords'
    ).load()

    # Extract wind components
    point_u = ds['u100'].values
    point_v = ds['v100'].values

    # Compute wind speed
    speed = np.sqrt(point_u**2 + point_v**2)
    # Store the yearly mean wind speed
    year_wind_speed[year] = speed.mean()
    quantile_wind_speed[year] = np.percentile(speed, 10)

# Replace simulation period years with the mean wind speeds
X[:, 5] = np.vectorize(year_wind_speed.get)(X[:, 5])


# Get power curve value at 9 m/s wind speed
power_at_9 = []
for x, pc in enumerate(X[:,3]):
    power_at_9.append(power_curves.iloc[int(pc),18])
    
X[:,3] = power_at_9

Use input (X) - output (Y) dataset to compute sensitivity indicies 

In [ ]:
# Define conditioning intervals for each input variable 
n = [2, 4, 5, 9, 5, 5]
# Since we are computing a dummy variable we must add this to the labels 
X_Labels = [ 'Wind \nHeight', 'Timestep', 'Alpha','Power \nCurve', 'Hub-\nheight', 'Simulation \nPeriod', 'Dummy']

Nboot = 1000

# Compute sensitivity indices for Nboot bootstrap resamples

KS_median, KS_mean, KS_max, KS_dummy = PAWN.pawn_indices(X, Y[:,0], n, Nboot=Nboot, dummy = True)
KS_max = np.column_stack((KS_max, KS_dummy))
# KS_median and KS_mean and KS_max have shape (Nboot, M)
# Compute mean and confidence intervals of the sensitivity indices across the
# bootstrap resamples:
KS_max_m, KS_max_lb, KS_max_ub = aggregate_boot(KS_max) # shape (M,)

Plot sensitivity indicies 

In [ ]:
bar_colors = ['skyblue'] * 5 + ['mediumpurple']

# Single subplot for the energy case study
fig_energy, ax = plt.subplots(figsize=(12, 8))
fig_energy.suptitle('Energy Case Study', fontsize=22, fontweight='bold')

# Plot (offshore)
ax.set_title(r"$\bf{}$ Offshore", fontsize=20)
x = np.arange(len(X_Labels) - 1)
heights = KS_max_m[:-1]
yerr = [KS_max_m[:-1] - KS_max_lb[:-1], KS_max_ub[:-1] - KS_max_m[:-1]]
ax.bar(x, heights, yerr=yerr, capsize=6, color=bar_colors, edgecolor='k')

ax.set_xticks(x)
ax.set_xticklabels(X_Labels[:-1], rotation=0, ha='center', fontsize=14)
ax.set_ylabel('Sensitivity Index', fontsize=16)
ax.set_ylim(0, 0.8)
ax.grid(axis='y', linestyle='--', alpha=0.5)

for xpos in [4.5, 5.5]:
    ax.axvline(xpos, color='k', linestyle='--', linewidth=1)

# Shade region below onshore dummy threshold
threshold = KS_max_m[-1]
ax.axhspan(0, threshold, color='gray', alpha=0.2, zorder=0)

ax.tick_params(axis='y', labelsize=14)

custom_legend = [
    Patch(facecolor='skyblue', edgecolor='k', label='System relationship (R)'),
    Patch(facecolor='mediumpurple', edgecolor='k', label='External forcing (X)'),
    Patch(facecolor='lightgrey', edgecolor='k', label='Undetectable sensitivity'),
]

ax.legend(handles=custom_legend, loc='upper right', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()